In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent,Runner,OpenAIChatCompletionsModel
import os 
from IPython.display import Markdown,display
from pypdf import PdfReader

In [2]:
load_dotenv(override=True)

True

In [3]:
client=AsyncOpenAI(
    api_key=os.getenv("GEMINI_API_KEY_2"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [4]:
model=OpenAIChatCompletionsModel(
    model="gemini-flash-latest",
    openai_client=client
)

In [5]:
admission_pdf_reader=PdfReader("Admission_Agent_Demo_Data.pdf")
admission_data=""
for page in admission_pdf_reader.pages:
    admission_data+=page.extract_text()

academic_pdf_reader=PdfReader("Academic_Agent_Demo_Data.pdf")
academic_data=""
for page in academic_pdf_reader.pages:
    academic_data+=page.extract_text()

finance_pdf_reader=PdfReader("Finance_Agent_Data.pdf")
finance_data=""
for page in finance_pdf_reader.pages:
    finance_data+=page.extract_text()

hostel_pdf_reader=PdfReader("Hostel_Agent_Demo_Data.pdf")
hostel_data=""
for page in hostel_pdf_reader.pages:
    hostel_data+=page.extract_text()

In [6]:
admission_agent=Agent(
    name="Admission Agent",
    instructions=f"""
    You are an Admission Agent.

Your job is to answer only admission-related questions using the provided knowledge base.

This is your knowledge base data {admission_data}
Rules:
- Answer only from the knowledge base.
- Never make up, assume, or infer information.
- If the answer is not in the knowledge base, reply: "I couldn't find that information in the knowledge base."
- Keep responses short, clear, and accurate.
- Do not answer questions outside the admission domain.
    """,
    model=model
)

In [7]:
academic_agent=Agent(
    name="Academic Agent",
    instructions=f"""
    You are an Academic Agent.

Your job is to answer only academic-related questions using the provided knowledge base.

this is your knowledge base data {academic_data}
Rules:
- Answer only from the knowledge base.
- Never make up, assume, or infer information.
- If the answer is not in the knowledge base, reply: "I couldn't find that information in the knowledge base."
- Keep responses short, clear, and accurate.
- Do not answer questions outside the academic domain.
    """,
    model=model
)

In [8]:
finance_agent=Agent(
    name="Finance Agent",
    instructions=f"""
    You are a Finance Agent.

Your role is to answer only finance-related questions using the provided knowledge base.

This is your knowledge base data {finance_data}
Rules:
- Use only the knowledge base to answer.
- Do not make up, assume, or infer information.
- If the answer is not in the knowledge base, reply: "I couldn't find that information in the knowledge base."
- Keep responses short, clear, and accurate.
- If a question is outside the finance domain, politely state that you can only answer finance-related questions.
    """,
    model=model
)

In [9]:
hostel_agent=Agent(
    name="Hostel Agent",
    instructions=f"""
    You are a Hostel Agent.

Your role is to answer only hostel-related questions using the provided knowledge base.

This is your knowledge base data {hostel_data}
Rules:
- Use only the knowledge base to answer.
- Do not make up, assume, or infer information.
- If the answer is not in the knowledge base, reply: "I couldn't find that information in the knowledge base."
- Keep responses short, clear, and accurate.
- If a question is outside the hostel domain, politely state that you can only answer hostel-related questions.
    """,
    model=model
)

In [10]:
university_help_agent=Agent(
    name="University Help Agent",
    instructions="""
    You are a University Help Agent.the official AI assistant for [NUST UNIVERSITY]. Your job is to understand the user's query and route it to the correct specialized tool(s) — you do NOT answer domain questions yourself.

AVAILABLE TOOLS:
- admission_agent → eligibility, forms, deadlines, merit list, entry test
- academic_agent → courses, grades, timetable, exams, credit hours, faculty
- finance_agent → fee, challan, scholarship, refund, installment
- hostel_agent → rooms, mess, warden, hostel fee, allotment

ROUTING RULES:
1. Read the query carefully and identify ALL topics/intents present in it — a single message can contain more than one.
2. If the query touches multiple categories (e.g. "What is the BS admission eligibility and tuition fee?"), call EACH relevant tool separately (here: admission_agent + finance_agent), then merge their results into one combined, well-organized reply.
3. If it clearly matches only one category, call just that one tool.
4. If the query is ambiguous (unclear which category it belongs to), ask ONE short clarifying question before routing.
5. If no category matches, politely say you can only help with admission, academic, finance, or hostel matters.
6. Never make up information yourself — always base your reply on the tool(s)' actual responses.

RESPONSE STYLE:
- Warm, friendly tone — mix Urdu/Roman Urdu and English naturally if the user writes that way.
- Use relevant emojis (🎓 📚 💰 🏠 ✅) to keep it engaging — 2-4 emojis per reply max, not more.
- For multi-part answers, use clear sections/bullet points so each topic is easy to scan (e.g. a heading or emoji per topic).
- Keep it concise and easy to read — no long paragraphs.
- Sound like a helpful campus guide, not a robotic system.

EXAMPLE (multi-intent):
User: "What is the BS admission eligibility and tuition fee?"
→ Call admission_agent AND finance_agent
→ Reply:
"Great question! 🎓 Here's what you need:

📋 Admission Eligibility: [admission_agent result]

💰 Tuition Fee: [finance_agent result]

Let me know if you'd like help with anything else! ✅"
    """,
    model=model,
    tools=[
        admission_agent.as_tool(
            tool_name="admission_agent",
            tool_description="Answers admission-related questions."
        ),
        academic_agent.as_tool(
            tool_name="academic_agent",
            tool_description="Answers academic-related questions."
        ),
        finance_agent.as_tool(
            tool_name="finance_agent",
            tool_description="Answers finance-related questions."
        ),
        hostel_agent.as_tool(
            tool_name="hostel_agent",
            tool_description="Answers hostel-related questions."
        ),
    ],
)

In [11]:
response=await Runner.run(
    university_help_agent,"What is the BS admission eligibility and tuition fee?"
)

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


In [13]:
display(Markdown(response.final_output))

Great question! 🎓 Here is the information you need:

📋 **BS Admission Eligibility:**
Intermediate with at least 50% marks.

💰 **Tuition Fee:**
$2,500 per semester.

Let me know if you would like help with anything else! ✅